# Add UIDs to GeoJSON

Injects a `uid` field (a 63-bit cryptographically random integer) into every feature of an existing GeoJSON file, **in-place**, using GDAL/OGR.

The `uid` field is required by `_makepatch_geojson` when loading patches into the database.  Each UID is generated with `secrets.randbelow(2**63)`, producing a value that fits in a PostgreSQL `BIGINT` column and is practically collision-free at any realistic dataset size.

> **Run `add_uids_to_geojson` before `init1` if your GeoJSON does not already contain a `uid` field.**

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit this cell before running
# ---------------------------------------------------------------------------

GEOJSON_PATH = "path/to/patches.geojson"  # modified in-place

In [ ]:
import secrets

from osgeo import ogr, osr

# Open the datasource in update mode (1 = update).
datasource = ogr.Open(GEOJSON_PATH, 1)
if datasource is None:
    raise RuntimeError(f"OGR could not open: {GEOJSON_PATH}")

layer = datasource.GetLayer(0)

# Add the uid field if it does not already exist.
layer_defn = layer.GetLayerDefn()
existing_fields = {layer_defn.GetFieldDefn(i).GetName() for i in range(layer_defn.GetFieldCount())}

if "uid" not in existing_fields:
    uid_field = ogr.FieldDefn("uid", ogr.OFTInteger64)
    layer.CreateField(uid_field)
    print("Created 'uid' field.")
else:
    print("'uid' field already exists — values will be overwritten.")

# Write a unique random 63-bit integer to every feature.
layer.ResetReading()
count = 0
for feature in layer:
    uid_val = secrets.randbelow(2**63)
    feature.SetField("uid", uid_val)
    layer.SetFeature(feature)
    count += 1

# Flush writes to disk.
datasource.FlushCache()
datasource = None

print(f"UIDs written to {count} features in {GEOJSON_PATH}")